**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Real-Time Signal Processing

The workshop [Intro to GPU Systems](../Intro_GPU/Intro_GPU.ipynb) promised: what changes when the signal *keeps coming* and every block has a **deadline**. Fixed-point arithmetic, block processing under a latency budget, and a simulated real-time pipeline with measured deadline misses — the glue between [DSP](./README.md), [OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb), and [FPGA](../Intro_FPGA/README.md).

## 1. Pre-requisites

- [Filter Design](./Filter_Design.ipynb), [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) (block processing).
- [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — scheduling jitter is the enemy here.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
import time
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Fixed-Point Arithmetic* (~40 min)
**Goal:** represent signals in Q-format; measure quantization noise; watch overflow bite.
**Builds on:** [Intro to C](../Intro_Programming/Intro_C.ipynb) (bits). &nbsp; **Feeds into:** Session 2 (latency budgets).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Fixed-Point Arithmetic</b></summary>

**Timing (~40 min).** 8 min why integer math at all · 10 min Q-format at the board · 7 min the SNR demo and its 3 dB surprise · 12 min the overflow demo · 3 min buffer. The overflow demo is the one students remember; protect its time.

**Open with the "why."** Students raised on `float` think fixed point is a historical curiosity. It is not: it is what runs on the MCU in their earbuds and in [FPGA fabric](../Intro_FPGA/Intro_FPGA.ipynb), where a multiplier is silicon you pay for. Ask who has heard a device glitch, crackle, or wrap into noise — Session 1 explains both failure sounds.

**Board first.** Write `0110 1000 0000 0000` and ask what number it is. The answer is "you can't know" — the binary point is a convention in the programmer's head, not in the register. That is Q-format in one image, and it is why every multiply needs its `>> 15`.

**Misconception.** "Quantization noise and overflow are both just error." They could not behave more differently. Quantization is graceful and additive — you lose 6 dB of SNR per bit dropped and the signal gets hissy. Overflow is catastrophic and non-linear — the largest positive value wraps to the largest *negative* one, so the loudest part of the signal becomes the most broken. Make them predict what each sounds like before running the second cell.

**Ask the room.** Before running the SNR cell: "we drop 8 bits — how much SNR do we lose?" Get them to commit to a number. The 6 dB-per-bit rule predicts about 48 dB, and the measurement lands there. A prediction they made themselves is worth more than a table they read.

**Watch for the 3 dB gap.** Every row measures ~3 dB *better* than the printed rule of thumb, and sharp students will catch it. This is not sloppiness — it is a difference in what "full scale" means, and the debrief below works it out. If nobody notices, point at it yourself: it is a better lesson than the rule itself.

**If the demo misbehaves.** The overflow cell is deliberately written the wrong way and is slow (an explicit Python loop) — expect a few seconds. If `y_bad` comes out looking *fine*, the input amplitude has been lowered; the products need to be large enough to actually wrap.
</details>

## 2. Numbers Without a Float Unit

💡 **Intuition.** Microcontrollers and [FPGA fabric](../Intro_FPGA/Intro_FPGA.ipynb) do integer math. **Q-format** fakes fractions with an implicit binary point: Q1.15 stores $x \in [-1, 1)$ as $\mathrm{round}(x \cdot 2^{15})$ in an int16. Each quantization adds ~uniform noise of variance $\Delta^2/12$ — *6 dB of SNR per bit* — and every multiply must be re-scaled (>> 15) or the binary point drifts. The two failure modes to respect: **quantization noise** (graceful, hissy) and **overflow** (catastrophic, wrap-around).

In [2]:
def to_q15(x):  return np.clip(np.round(x * 2**15), -2**15, 2**15 - 1).astype(np.int16)
def from_q15(q): return q.astype(np.float64) / 2**15

t = np.arange(0, 1, 1/8000)
x = 0.7 * np.sin(2*np.pi*440*t)

for bits in [15, 11, 7]:
    scale = 2**bits
    q = np.round(x * scale) / scale
    snr = 10*np.log10(np.var(x) / np.var(x - q))
    print(f"Q1.{bits:2d}: measured SNR {snr:5.1f} dB   (rule of thumb ≈ 6.02×{bits}+1.76 = {6.02*bits+1.76:5.1f} dB)")

Q1.15: measured SNR  95.2 dB   (rule of thumb ≈ 6.02×15+1.76 =  92.1 dB)
Q1.11: measured SNR  71.4 dB   (rule of thumb ≈ 6.02×11+1.76 =  68.0 dB)
Q1. 7: measured SNR  46.9 dB   (rule of thumb ≈ 6.02×7+1.76 =  43.9 dB)


**What just happened.** Drop 4 bits and you lose about 24 dB; drop 8 and you lose about 48. The measured column falls by **23.8 dB** then **24.5 dB** as we go 15 → 11 → 7 bits, which is the 6.02 dB-per-bit rule confirmed on a real signal. This is the number to carry around: *one bit is 6 dB*, so an audio codec at 16 bits has ~96 dB of dynamic range and a 12-bit ADC gives you ~72.

**Now the interesting part.** Every row beats the printed rule of thumb by almost exactly 2.9 dB. That is not measurement luck — it is a reminder that $\mathrm{SNR} = 6.02b + 1.76$ carries an assumption most textbooks state once and never repeat: it is for a sine that *fills the full scale* of a converter whose range spans $2^b$ steps. Our setup differs in two ways that nearly cancel:

- our sine has amplitude 0.7, not 1.0, which costs $20\log_{10}(0.7) = -3.1$ dB;
- our Q-format puts $b$ fractional bits across $[-1, 1)$, so the step is $2^{-b}$ rather than $2/2^{b}$ — a factor of two finer, worth $+6.02$ dB.

Net: $+2.9$ dB, matching the gap in every row. Redo the arithmetic with our actual amplitude and step size and the prediction becomes 95.0 / 70.9 / 46.8 dB against the measured 95.2 / 71.4 / 46.9. The theory was never wrong; we were quoting it outside its assumptions.

The lesson generalizes past this cell. A rule of thumb that disagrees with your measurement by a constant offset is almost always a units or full-scale convention mismatch, not a broken system — chase the constant before you chase the code.

In [3]:
# A Q15 FIR filter, and the overflow trap
h = sig.firwin(31, 0.2)
h_q = to_q15(h)
x_q = to_q15(x)

# CORRECT: accumulate in int32 (headroom!), shift back once
acc = np.convolve(x_q.astype(np.int64), h_q.astype(np.int64))            # 32-bit safe products
y_good = from_q15(np.clip(acc >> 15, -2**15, 2**15 - 1).astype(np.int16))

# WRONG: no headroom — products wrap in int16
y_bad = np.zeros(len(x_q))
prod = (x_q.astype(np.int32)[:, None] * h_q.astype(np.int32)[None, :]) >> 15
prod16 = prod.astype(np.int16)                                            # silent wraparound!
for i_ in range(31, len(x_q)):
    y_bad[i_] = from_q15(np.sum(prod16[i_-30:i_+1, ::-1].diagonal()).astype(np.int16))

y_ref = np.convolve(x, h)[:len(x)]
plt.figure(figsize=(9, 2.6))
plt.plot(y_ref[400:600], "k--", linewidth=1, label="float reference")
plt.plot(y_good[400:600], label="Q15, int32 accumulator: indistinguishable")
plt.plot(y_bad[400:600], alpha=0.6, label="Q15, int16 accumulation: overflow chaos")
plt.legend(fontsize=8); plt.title("headroom is not optional")
plt.tight_layout(); plt.show()
print(f"error vs float — with headroom: {np.abs(y_good[:len(x)]-y_ref).max():.2e}   without: {np.abs(y_bad-y_ref).max():.2f}")

error vs float — with headroom: 5.52e-05   without: 0.68


/tmp/ipykernel_2703251/2941568280.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Same filter, same input, same Q15 format — and a factor of **twelve thousand** difference in error: `5.52e-05` with an int32 accumulator versus `0.68` without. The orange trace is unusable, and notice *how* it fails: not as added hiss but as violent excursions that appear exactly where the signal is largest, because that is where products overflow and wrap from large-positive to large-negative.

The cause is arithmetic, not DSP. Multiplying two Q15 numbers produces a Q30 result, which needs 32 bits before you shift it back down to 16 — and an FIR sums 31 of those products, so the accumulator needs headroom for the sum as well. The correct version accumulates wide and shifts *once* at the end. The broken version narrows to int16 after every multiply, and the wraparound is silent: no exception, no warning, just a wrong number that keeps flowing downstream.

This is the defining hazard of fixed-point work. A float pipeline that goes wrong usually announces itself with `nan` or `inf`; an integer pipeline hands you a plausible-looking int16 and lets you ship it. The discipline that prevents it is fixed: accumulate wide, shift late, saturate rather than wrap at the boundaries (`np.clip` here is standing in for the saturating instruction a DSP core gives you in hardware).

---
### 🕐 Session 2 of 3 — *Latency Budgets & Block Processing* (~35 min)
**Goal:** count the milliseconds: block size sets the latency floor; compute must fit inside it.
**Builds on:** Session 1; [OS workshop](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb). &nbsp; **Feeds into:** Session 3 (a real-time pipeline).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Latency Budgets & Block Processing</b></summary>

**Timing (~35 min).** 10 min the two laws at the board · 8 min the block-size trade-off · 10 min the timing table · 7 min buffer. Shortest session of the three — leave room, because the table invites good questions.

**Board first.** Draw the two-block timeline: block $k$ being processed while block $k{+}1$ records. Everything in this session is read off that one picture — the latency floor because you cannot output before a block has filled, and the throughput wall because the processing bar must fit under the recording bar. Do not write a formula until the picture is on the board.

**Make it concrete.** Ask who has turned down the buffer size in a DAW, a game, or a streaming setup to reduce lag, and what happened. Someone will have hit crackling. That crackle *is* the throughput wall: a smaller buffer bought lower latency until compute stopped fitting inside it. This session's table is that knob, measured.

**Misconception.** "Faster hardware fixes deadline misses." It raises the ceiling but changes nothing structurally: the latency floor is set by block size and sample rate alone — $B/f_s$ — and no CPU makes a 64-sample block at 48 kHz arrive sooner than 1.33 ms. Latency and throughput are separate budgets, and students routinely collapse them into one idea of "speed."

**Ask the room.** "Which column wins at 64 samples, and which at 4096?" Have them predict before the table prints. The direct convolution winning at small blocks surprises people who have internalized "FFT is faster" — the FFT's asymptotic advantage does not pay for its constant overhead until the block is big enough.

**Honest framing of the result.** Every row passes on a modern laptop; nothing misses. Do not oversell it as a near-miss — the real content is the *crossover* between the two algorithms and the shrinking margin, and the debrief below says so plainly. The genuine deadline pressure arrives in Session 3 and, more honestly, on the embedded targets students will actually deploy to.
</details>

## 3. The Budget

💡 **Intuition.** A real-time system processes block $k$ while block $k{+}1$ records. Two laws follow. **Latency floor:** you can't output before a block fills — latency ≥ one block (plus compute, plus output buffering); small blocks = low latency. **Throughput wall:** compute per block must finish in under one block-duration — or you fall behind *forever*. Small blocks also mean more per-block overhead ([OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) syscalls, scheduling), so the budget squeezes from both sides. Every audio interface's 'buffer size' knob is exactly this dial.

In [4]:
fs = 48_000
h_long = sig.firwin(2049, 0.1)                       # a deliberately heavy filter
x_blk = rng.standard_normal(fs)                      # 1 second of audio

print(f"{'block':>6} {'deadline':>9} {'direct conv':>12} {'FFT (overlap-save)':>19}")
for B in [64, 256, 1024, 4096]:
    deadline_ms = B / fs * 1000
    xb = x_blk[:B]
    tic = time.perf_counter()
    for _ in range(20): np.convolve(xb, h_long)
    t_direct = (time.perf_counter() - tic) / 20 * 1000
    tic = time.perf_counter()
    for _ in range(20): sig.oaconvolve(xb, h_long)
    t_fft = (time.perf_counter() - tic) / 20 * 1000
    verdict = lambda t: "✓" if t < deadline_ms else "✗ MISSES"
    print(f"{B:>6} {deadline_ms:>7.2f}ms {t_direct:>9.3f}ms {verdict(t_direct):8s} {t_fft:>9.3f}ms {verdict(t_fft)}")
print("\n→ the fast-convolution algorithms of [Foundations 1 S8] are what make small deadlines feasible")

 block  deadline  direct conv  FFT (overlap-save)
    64    1.33ms     0.015ms ✓            0.133ms ✓
   256    5.33ms     0.038ms ✓            0.052ms ✓
  1024   21.33ms     0.122ms ✓            0.055ms ✓
  4096   85.33ms     0.468ms ✓            0.084ms ✓

→ the fast-convolution algorithms of [Foundations 1 S8] are what make small deadlines feasible


**What just happened.** Two things, and it is worth separating them because only one is about deadlines.

**The latency floor is pure arithmetic.** A 64-sample block at 48 kHz takes 1.33 ms to fill, and 4096 samples take 85.3 ms. No amount of compute speed changes that column — it is $B/f_s$ and nothing else. This is why low-latency audio interfaces advertise small buffers, and why a 4096-sample buffer is unusable for a musician monitoring themselves live (85 ms of delay is roughly a person standing 30 metres away) but perfectly fine for offline rendering.

**The algorithm crossover is the real find.** Direct convolution wins decisively at 64 samples (0.015 ms vs 0.133 ms — nearly 9× faster than the FFT), the two roughly tie by 256, and the FFT pulls ahead at 1024 and beyond (0.055 ms vs 0.122 ms, then 0.084 ms vs 0.468 ms). Note what the FFT column *does*: it barely moves from 64 to 4096 while direct convolution's cost climbs with block size. That is $O(N \log N)$ against $O(N \cdot M)$ made visible, and it is why the fast-convolution machinery from Foundations 1 exists.

**Be honest about the verdicts.** Every configuration passes here — no `✗ MISSES` anywhere — because a 2049-tap filter on one second of audio is not much work for a modern laptop. Don't read this table as "real-time is easy." Read it as the *shape* of the budget on a fast machine with nothing else running. Three things collapse the margin in practice: an embedded target 100× slower, a system doing many channels at once rather than one, and an operating system that takes the CPU away at the wrong moment. Session 3 measures that last one.

Keep an eye on the ratio rather than the absolute numbers. At 1024 samples, FFT convolution uses 0.055 ms of a 21.3 ms budget — about 0.3% — and that fraction, not the millisecond count, is what tells you whether the system survives a bad day.

---
### 🕐 Session 3 of 3 — *A Real-Time Pipeline, Simulated & Measured* (~40 min)
**Goal:** producer/consumer with deadlines; measure misses and jitter like an engineer.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: A Real-Time Pipeline, Simulated & Measured</b></summary>

**Timing (~40 min).** 8 min the producer/consumer architecture · 7 min what to measure and why · 10 min the run and its histogram · 10 min the load experiment (below — this is the session's real payoff) · 5 min buffer.

**Connect it back.** This is the [OS workshop's](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) producer/consumer with one addition: a clock. Students have seen the queue, the bounded buffer, the two threads. What is new is that a slow consumer is no longer merely inefficient — it is *incorrect*, because audio that arrives late is as useless as audio that never arrives. Say that sentence out loud; it is the definition of a real-time system and it is what separates this session from a threading tutorial.

**Why `lfilter` carries `zi`.** Worth ninety seconds. Filtering each block independently would produce a click at every boundary, because the filter's memory resets. The `zi` state threaded through the loop is what makes 300 separate blocks equal one continuous filtered signal — the same continuity concern as overlap-save in Session 2, wearing different clothes.

**Run the load experiment — do not skip it.** As shipped, this cell reports zero misses and a histogram in a tight, boring spike, because the machine is idle and the filter is cheap. That result is *correct and unconvincing*, and students learn nothing about jitter from a clean run. Re-run it with the machine under real load: start a training job (the [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) notebook works well), or in a terminal run a busy loop on every core — on Linux, `stress-ng --cpu $(nproc)`, or failing that several `yes > /dev/null &` processes. The histogram grows a tail toward the deadline line. *That* is the picture worth recording.

**Ask the room.** "We had 21 ms of headroom and zero misses. Are we safe to ship?" The answer is no, and the reason is the shape of the distribution rather than its centre: a system whose worst case sits close to the deadline will miss as soon as anything unusual happens, and the average tells you nothing about that. Engineers size real-time systems by the tail, which is why we plot a histogram instead of printing a mean.

**Honest caveat to state plainly.** Python's GIL and `time.sleep` granularity make this a *simulation* of real-time behaviour, not a real-time system. No one ships audio this way. What transfers is the methodology — deadline per block, measure headroom, look at the worst case — not the implementation. If a student asks how it is really done, that is the cue for the callbacks-and-lock-free-ring-buffers conversation, and for [FPGA](../Intro_FPGA/Intro_FPGA.ipynb) when even that jitter is too much.
</details>

## 4. The Pipeline

Architecture (the [OS workshop's](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) producer/consumer, with a clock): an acquisition thread produces blocks on schedule; a processing thread must consume + filter each block before the next arrives. We measure the **headroom histogram** — the engineer's dashboard for 'will this survive a bad scheduling day?'

In [5]:
import threading, queue, collections

fs2, B = 48_000, 1024
block_dur = B / fs2
h_rt = sig.firwin(513, 0.1)
n_blocks = 300

q_ = queue.Queue(maxsize=8)
headroom, misses = [], 0
DONE = object()

def producer():
    next_t = time.perf_counter()
    for k in range(n_blocks):
        next_t += block_dur
        blk = rng.standard_normal(B)
        q_.put((blk, next_t))                      # deadline: before the NEXT block lands
        sleep = next_t - time.perf_counter()
        if sleep > 0: time.sleep(sleep)
    q_.put(DONE)

def consumer():
    global misses
    zi = np.zeros(len(h_rt) - 1)
    while (item := q_.get()) is not DONE:
        blk, deadline = item
        _, zi = sig.lfilter(h_rt, 1, blk, zi=zi)   # stateful streaming filter
        slack = deadline - time.perf_counter()
        headroom.append(slack * 1000)
        if slack < 0: misses += 1

t1, t2 = threading.Thread(target=producer), threading.Thread(target=consumer)
t1.start(); t2.start(); t1.join(); t2.join()

hr = np.array(headroom)
plt.figure(figsize=(8, 2.6))
plt.hist(hr, bins=60)
plt.axvline(0, color="r", linewidth=1.5, label="deadline")
plt.xlabel("headroom at completion [ms]"); plt.legend()
plt.title(f"headroom histogram over {n_blocks} blocks — {misses} deadline misses")
plt.tight_layout(); plt.show()
print(f"block budget {block_dur*1000:.1f} ms | median headroom {np.median(hr):.2f} ms | worst {hr.min():.2f} ms | misses {misses}")
print("engineering rule: if the worst-case headroom is a small fraction of the budget, you WILL glitch")
print("under load — re-run this while the Scale_NN notebook trains to see the OS steal your margin.")

block budget 21.3 ms | median headroom 21.07 ms | worst 20.81 ms | misses 0
engineering rule: if the worst-case headroom is a small fraction of the budget, you WILL glitch
under load — re-run this while the Scale_NN notebook trains to see the OS steal your margin.


/tmp/ipykernel_2703251/1821338709.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** 300 blocks, a 21.3 ms budget each, **zero deadline misses**, with median headroom 21.07 ms and a worst case of 20.81 ms. Filtering a block cost roughly 0.25 ms — about 1% of the budget — so the consumer spends almost all its time waiting for the producer, which is exactly what a healthy real-time system looks like.

**Now read the histogram the way an engineer would.** The number that matters is not the median, it is the **worst case**: 20.81 ms of headroom, so even the slowest block finished with 98% of its budget to spare. The distribution is tight, and tightness is the property you are shopping for — a real-time system is sized by its tail, because the deadline is missed by the worst block, never the average one. A system with 15 ms of *median* headroom but a worst case of 0.2 ms is far more dangerous than this one, and no summary statistic except the minimum would tell you.

**And be suspicious of this result.** It is clean because the machine was idle and the filter is cheap; it demonstrates the measurement method, not a hard scheduling problem. Two caveats to keep:

- **The load caveat.** The margin above belongs to an unloaded machine. Re-run this cell while something heavy competes for the CPU and the histogram grows a tail stretching toward the red line — the operating system's scheduler takes your margin without asking. That tail is the real subject of this session.
- **The platform caveat.** Python's GIL and `time.sleep` granularity mean this simulates real-time behaviour rather than achieving it. Production audio uses callbacks driven by the sound card's own clock and lock-free ring buffers, precisely to avoid the jitter sources we are subject to here.

What transfers is the discipline, not the code: give every block a deadline, measure headroom at completion, and judge the system by its worst block. When the tail crosses the line and it cannot be fixed in software, the escape hatches from the conclusion are what remain — [GPU batching](../Intro_GPU/README.md) for throughput, [FPGA](../Intro_FPGA/Intro_FPGA.ipynb) for deterministic latency.

## 5. Conclusion

Six dB per bit, headroom before shifting, latency ≥ one block, compute < one block-duration, and always look at the *worst-case* headroom, not the average. When the budget can't be met on a CPU, you now know both escape hatches: [GPU batching](../Intro_GPU/README.md) (throughput, at latency cost) and [FPGA](../Intro_FPGA/Intro_FPGA.ipynb) (deterministic latency, at effort cost).

---
## Where next

- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — the same Q15 FIR as literal hardware.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — MHz-rate streams where these budgets get serious.
- [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — the scheduler that owns your jitter.